In [10]:
   #@title
from IPython.display import HTML
HTML('<div style="height:24px;background:repeating-linear-gradient(-45deg,#F2C230 0 16px,#14232E 16px 32px)"></div>')



<h1 align="center">The Canteen Ledger That Lied</h1>
<p align="center">A data cleaning case file for DS311 Exploratory Data Analysis, Week 2: Data Wrangling and Data Cleaning</p>

***

Nobody broke into the canteen. Nothing was stolen. But the weekly sales report says one customer bought **500 siomai** in a single order, and the owner swears that never happened. Or did it?

Most lessons on data cleaning start with a list of commands. I want to start with a question instead: **what would a detective do?** A detective never arrests the first suspicious person. They collect evidence, ask why something looks wrong, and only then decide what to do about it.

That is the mindset behind data cleaning. So let's open the case. *(The ledger is made up for this story, but every problem in it is one you will meet in real data.)*

## Exhibit A: the ledger

A school canteen records its orders in a spreadsheet. Here is the raw export, exactly as the cashier typed it. The red cells are the clues I want to investigate.

In [11]:
#@title
import pandas as pd

data = {
    "Order": [1001, 1002, 1003, 1004, 1005, 1006, 1007, 1008],
    "Item": ["Siomai", "siomai", "Koykoy", "Koykoy", "Siomai", "SIOMAI", "Rice Meal", "Rice Meal"],
    "Qty": [3, 2, 1, 1, -2, 500, 1, 2],
    "Price": [25, 25, 30, 30, 25, 25, None, 55],
    "Date": ["2026-09-01", "09/01/2026", "2026-09-02", "2026-09-02", "September 3, 2026", "2026-09-04", "2026-09-04", "09/05/2026"],
    "Payment": ["Cash", "GCash", "Cash", "Cash", "Cash", "gcash", "Cash", "Cash"],
}

df = pd.DataFrame(data)

# cells to highlight as clues: (row, column)
flags = [(1,"Item"),(1,"Date"),(3,"Order"),(4,"Qty"),(4,"Date"),
         (5,"Item"),(5,"Qty"),(5,"Payment"),(6,"Price"),(7,"Date")]

def paint(d):
    out = pd.DataFrame("", index=d.index, columns=d.columns)
    for r, c in flags:
        out.loc[r, c] = "background-color:#F6D9D5; color:#C8372D; font-weight:bold"
    return out

df.style.apply(paint, axis=None).format({"Price": "{:.0f}"}, na_rep="(empty)")

,Order,Item,Qty,Price,Date,Payment
0,1001,Siomai,3,25,2026-09-01,Cash
1,1002,siomai,2,25,09/01/2026,GCash
2,1003,Koykoy,1,30,2026-09-02,Cash
3,1004,Koykoy,1,30,2026-09-02,Cash
4,1005,Siomai,-2,25,"September 3, 2026",Cash
5,1006,SIOMAI,500,25,2026-09-04,gcash
6,1007,Rice Meal,1,(empty),2026-09-04,Cash
7,1008,Rice Meal,2,55,09/05/2026,Cash


## First look: what does the evidence say?

Before changing anything, a detective just looks. Two quick commands tell us a lot.

In [12]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8 entries, 0 to 7
Data columns (total 6 columns):
 #   Column   Non-Null Count  Dtype  
---  ------   --------------  -----  
 0   Order    8 non-null      int64  
 1   Item     8 non-null      object 
 2   Qty      8 non-null      int64  
 3   Price    7 non-null      float64
 4   Date     8 non-null      object 
 5   Payment  8 non-null      object 
dtypes: float64(1), int64(2), object(3)
memory usage: 516.0+ bytes


**What I noticed:**

- `Price` has only 7 non-null values out of 8. One price is missing.
- `Date` is listed as `object`, which means pandas sees it as plain text, not as real dates. That is why the dates can be written in three different styles without any error.

In [13]:
df.describe()

,Order,Qty,Price
count,8.00000,8.000000,7.000000
mean,1004.50000,63.500000,30.714286
std,2.44949,176.378651,10.965313
min,1001.00000,-2.000000,25.000000
25%,1002.75000,1.000000,25.000000
50%,1004.50000,1.500000,25.000000
75%,1006.25000,2.250000,30.000000
max,1008.00000,500.000000,55.000000


**What I noticed:**

- `Qty` has a minimum of **-2** and a maximum of **500**. A negative quantity and a huge quantity are both worth investigating.
- The mean `Qty` is 63.5, but the median is only 1.5. One order is dragging the average far away from what a typical order looks like, so I shouldn't trust the mean alone.
- `describe()` even gave me an average order number (1004.5). That's meaningless, because an order number is just a label, not something you can average.

## Wrangling is the investigation. Cleaning is one step of it.

Many students use "data wrangling" and "data cleaning" as if they mean the same thing. In our detective story the difference is easy to see.

- **Data wrangling** is the whole investigation: getting the ledger, reading it, structuring it, fixing it, checking it, and handing over a report someone can trust.
- **Data cleaning** is the part where we examine each piece of evidence and correct or remove what is wrong.

All cleaning is wrangling, but not all wrangling is cleaning. Reshaping a table, or joining this ledger to a menu list, is wrangling even though nothing is "dirty."

## The five clues

Every clue follows the same routine: **spot it, ask why, then decide.**

### Clue 1: the twin orders

Orders 1003 and 1004 are both Koykoy, quantity 1, price 30, same date, same payment. Only the order number is different.

Are they a double entry, or two hungry students who ordered the same thing? Plain `drop_duplicates()` compares every column, including `Order`, so it would miss these two rows. To catch them, I told it to ignore `Order` by listing only the other columns in `subset=[...]`.

I decided it was a double entry, so I kept 1003 and dropped 1004. The order numbers are next to each other and everything else matches. In real life I would ask the cashier before deleting anything.

In [14]:
df = df.drop_duplicates(subset=["Item", "Qty", "Price", "Date", "Payment"]).copy()
df

,Order,Item,Qty,Price,Date,Payment
0,1001,Siomai,3,25.0,2026-09-01,Cash
1,1002,siomai,2,25.0,09/01/2026,GCash
2,1003,Koykoy,1,30.0,2026-09-02,Cash
4,1005,Siomai,-2,25.0,"September 3, 2026",Cash
5,1006,SIOMAI,500,25.0,2026-09-04,gcash
6,1007,Rice Meal,1,NaN,2026-09-04,Cash
7,1008,Rice Meal,2,55.0,09/05/2026,Cash


### Clue 2: one siomai, three disguises

"Siomai", "siomai" and "SIOMAI" look the same to a person, but pandas counts them as three different items. A report grouping sales by item would split one best seller into three small ones. The same goes for "GCash" and "gcash".

I standardized the spelling with `.str.strip()`, `.str.title()` and `.str.lower()`. Doing this before the duplicate check matters, because rows that are really the same can slip past it if they only differ in capital letters.


In [15]:
df["Item"] = df["Item"].str.strip().str.title()
df["Payment"] = df["Payment"].str.strip().str.lower().replace({"gcash": "GCash", "cash": "Cash"})
df

,Order,Item,Qty,Price,Date,Payment
0,1001,Siomai,3,25.0,2026-09-01,Cash
1,1002,Siomai,2,25.0,09/01/2026,GCash
2,1003,Koykoy,1,30.0,2026-09-02,Cash
4,1005,Siomai,-2,25.0,"September 3, 2026",Cash
5,1006,Siomai,500,25.0,2026-09-04,GCash
6,1007,Rice Meal,1,NaN,2026-09-04,Cash
7,1008,Rice Meal,2,55.0,09/05/2026,Cash


### Clue 3: dates in three outfits

The ledger has `2026-09-01`, `09/01/2026` and `September 3, 2026`. Converting them all to real dates with `pd.to_datetime` lets me sort, filter by week, and calculate gaps between orders.

One thing to watch for: is `09/05/2026` September 5 or May 9? The nearby rows show it is September, but with real data I would check how the file was made before trusting the conversion.


In [16]:
df["Date"] = pd.to_datetime(df["Date"], format="mixed")
df.info()
df

<class 'pandas.core.frame.DataFrame'>
Index: 7 entries, 0 to 7
Data columns (total 6 columns):
 #   Column   Non-Null Count  Dtype         
---  ------   --------------  -----         
 0   Order    7 non-null      int64         
 1   Item     7 non-null      object        
 2   Qty      7 non-null      int64         
 3   Price    6 non-null      float64       
 4   Date     7 non-null      datetime64[ns]
 5   Payment  7 non-null      object        
dtypes: datetime64[ns](1), float64(1), int64(2), object(2)
memory usage: 392.0+ bytes


,Order,Item,Qty,Price,Date,Payment
0,1001,Siomai,3,25.0,2026-09-01,Cash
1,1002,Siomai,2,25.0,2026-09-01,GCash
2,1003,Koykoy,1,30.0,2026-09-02,Cash
4,1005,Siomai,-2,25.0,2026-09-03,Cash
5,1006,Siomai,500,25.0,2026-09-04,GCash
6,1007,Rice Meal,1,NaN,2026-09-04,Cash
7,1008,Rice Meal,2,55.0,2026-09-05,Cash


### Clue 4: the missing price, and why the median would be wrong

Order 1007 (Rice Meal) has no price. A common move is to fill gaps with the median. But the median price in this ledger is 25, and a rice meal doesn't cost 25. Order 1008 shows that Rice Meal costs **55**.

So I filled the gap with the price from the same item. A missing value is often better fixed with a trustworthy source than with a statistic. I would use the median only when I have nothing better.


In [17]:
print(df["Price"].median())   # 25.0, which is wrong for a Rice Meal

df["Price"] = df["Price"].fillna(df.groupby("Item")["Price"].transform("first"))
df

25.0


,Order,Item,Qty,Price,Date,Payment
0,1001,Siomai,3,25.0,2026-09-01,Cash
1,1002,Siomai,2,25.0,2026-09-01,GCash
2,1003,Koykoy,1,30.0,2026-09-02,Cash
4,1005,Siomai,-2,25.0,2026-09-03,Cash
5,1006,Siomai,500,25.0,2026-09-04,GCash
6,1007,Rice Meal,1,55.0,2026-09-04,Cash
7,1008,Rice Meal,2,55.0,2026-09-05,Cash


### Clue 5: the two suspicious quantities

Order 1005 has a quantity of **-2** and order 1006 has a quantity of **500**. A beginner's rule might say "negative is impossible and 500 is an outlier, so delete both."

But an unusual value is a **lead, not a verdict.** In this story the cashier told me:

- The -2 was a **refund**. Two siomai were returned.
- The 500 was a **bulk order** for a school event.

Deleting the first would hide the refund, and deleting the second would erase the canteen's biggest sale. So I kept both, added an `is_refund` column for the refund, and added a "bulk order" note for the big one.


In [18]:
df["is_refund"] = df["Qty"] < 0
df.loc[df["Qty"] > 100, "note"] = "bulk order"
df

,Order,Item,Qty,Price,Date,Payment,is_refund,note
0,1001,Siomai,3,25.0,2026-09-01,Cash,False,NaN
1,1002,Siomai,2,25.0,2026-09-01,GCash,False,NaN
2,1003,Koykoy,1,30.0,2026-09-02,Cash,False,NaN
4,1005,Siomai,-2,25.0,2026-09-03,Cash,True,NaN
5,1006,Siomai,500,25.0,2026-09-04,GCash,False,bulk order
6,1007,Rice Meal,1,55.0,2026-09-04,Cash,False,NaN
7,1008,Rice Meal,2,55.0,2026-09-05,Cash,False,NaN


## The case log

A good detective writes down every decision. A cleaning log lets a teammate (or me, three weeks from now) understand and reverse any choice.

| Problem | Action | Why |
| :--- | :--- | :--- |
| Twin Koykoy orders 1003 and 1004 | Dropped 1004 and kept 1003, checking every column except `Order` | Same item, quantity, price, date and payment, and the order numbers are next to each other, so it looks like a double entry. In real life I would confirm with the cashier first. |
| Item and payment spelling | Standardized `Item` to "Siomai" and `Payment` to "Cash" or "GCash" | pandas counts "Siomai", "siomai" and "SIOMAI" as different items, which would split the sales of one product into three groups. |
| Mixed date formats | Converted `Date` to real dates with `pd.to_datetime` | The dates were plain text in three styles, so I couldn't sort or filter them. Now they all look like 2026-09-XX. |
| Missing price on order 1007 | Filled it with 55, the Rice Meal price from order 1008 | The median price is 25, which is wrong for a rice meal. A price from the same item is more trustworthy than a statistic. |
| Qty of -2 | Kept it and added an `is_refund` column | A negative quantity is probably a returned order. Deleting it would hide a real transaction. |
| Qty of 500 | Kept it and added a "bulk order" note | It is unusual but possible, such as an order for a school event. Deleting it would erase the canteen's biggest sale. |

## What the case taught me

The commands I used (`drop_duplicates`, `to_datetime`, `fillna`) are quick to learn. The hard part is the thinking around them: spot the problem, ask why it happened, choose the least harmful fix, and write it down. If the data can't answer the "why," ask the person who made it.

The ledger didn't lie on purpose. People type quickly, spell things differently, and sometimes record a refund as a negative number. The goal isn't to scold the data but to understand it well enough that my conclusions can be trusted.


**Case closed.**

***

*Written for DS311 Exploratory Data Analysis. The canteen ledger is a made-up dataset.*

*The draft text written are help from AI; the data, code, and final edits are mine*
